# NB04 v4 — Batched IDF lexical blocking

This S4 implementation follows the project decision: country-partitioned IDF-weighted token blocking over name and address. The key memory fix is batching the holdout Source-1 references; it never joins all 149,999 references to the 10M-row retrieval index at once.

In [ ]:
import hashlib, json, os, pathlib, sys, time
import pandas as pd
ON_SAGEMAKER = pathlib.Path('/opt/ml').exists() or 'SM_CURRENT_HOST' in os.environ
ROOT = pathlib.Path(os.environ.get('BER_ROOT', '/home/ec2-user/SageMaker/ber' if ON_SAGEMAKER else '/Users/pocketfm/Desktop/amazon_ML/Working Folder'))
DATA, ART = ROOT / 'student_resource/dataset', ROOT / 'artifacts'
ART.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / 'src'))
from business_entity_resolution.blocking_stream import write_index_parts
from business_entity_resolution.normalization_mined import load_mined_tables
mined = load_mined_tables(ART / 'normalization_v1.json')
normalization_fingerprint = hashlib.sha256(json.dumps(mined.to_dict(), sort_keys=True).encode()).hexdigest()
index_root = ART / 'nb04_index_v1'
index_manifest_path = index_root / 'manifest.json'
try:
    index_manifest = json.loads(index_manifest_path.read_text())
except FileNotFoundError:
    index_manifest = {}
needs_rebuild = index_manifest.get('normalization_fingerprint') != normalization_fingerprint
if needs_rebuild:
    print('normalization changed (or index is untracked): rebuilding compact index parts')
    index_root.mkdir(parents=True, exist_ok=True)
    index_counts = {}
    for source_name in ('train_source1', 'train_source2', 'train_source3'):
        rows, parts = write_index_parts(DATA / 'train' / f'{source_name}.tsv', index_root / source_name, mined=mined)
        index_counts[source_name] = {'rows': rows, 'parts': parts}
    index_manifest_path.write_text(json.dumps({'normalization_fingerprint': normalization_fingerprint, 'counts': index_counts}, indent=2) + '\n')
else:
    print('reusing index parts built with this normalization fingerprint')
import duckdb
con = duckdb.connect(database=str(ART / 'nb04_v4.duckdb'))
con.execute("PRAGMA threads=2")
con.execute("PRAGMA memory_limit='4GB'")
con.execute("PRAGMA max_temp_directory_size='4GB'")
(ART / 'nb04_v4_tmp').mkdir(parents=True, exist_ok=True)
spill = str(ART / 'nb04_v4_tmp').replace('\\', '\\\\').replace("'", "''")
con.execute("SET temp_directory = '" + spill + "'")
s1 = str(index_root / 'train_source1' / 'part-*.parquet')
s2 = str(index_root / 'train_source2' / 'part-*.parquet')
s3 = str(index_root / 'train_source3' / 'part-*.parquet')
con.execute("CREATE OR REPLACE TEMP TABLE holdout_ids AS SELECT entity_id FROM read_parquet(?)", [str(ART / 'holdout_s1.parquet')])
con.execute(f"""CREATE OR REPLACE TABLE retrieval_source AS
SELECT * FROM read_parquet('{s2}') UNION ALL SELECT * FROM read_parquet('{s3}')""")
con.execute("""CREATE OR REPLACE TABLE retrieval_tokens AS
SELECT entity_id, country, 'name' AS field, token FROM retrieval_source, UNNEST(string_split(name_key, chr(31))) AS u(token) WHERE token <> ''
UNION ALL
SELECT entity_id, country, 'address', token FROM retrieval_source, UNNEST(string_split(address_key, chr(31))) AS u(token) WHERE token <> ''
""")
con.execute("CREATE OR REPLACE TABLE token_stats AS SELECT country, field, token, COUNT(DISTINCT entity_id) AS df FROM retrieval_tokens GROUP BY country, field, token")
con.execute("CREATE OR REPLACE TABLE country_counts AS SELECT country, field, COUNT(DISTINCT entity_id) AS n FROM retrieval_tokens GROUP BY country, field")
con.execute(f"""CREATE OR REPLACE TABLE holdout_refs AS SELECT * FROM read_parquet('{s1}') WHERE entity_id IN (SELECT entity_id FROM holdout_ids)""")
print('retrieval index ready')

In [ ]:
# Safety default for the 8 GiB EC2 host: evaluate a stratified 5k-S1 pilot.
# Set BER_BLOCKING_EVAL_S1=0 only after the pilot has completed cleanly.
eval_limit = int(os.environ.get('BER_BLOCKING_EVAL_S1', '5000'))
batch_size = int(os.environ.get('BER_BLOCKING_BATCH_SIZE', '100'))
max_token_df = int(os.environ.get('BER_BLOCKING_MAX_DF', '5000'))
ks = (10, 20, 30, 50, 100)
all_holdout = pd.read_parquet(ART / 'holdout_s1.parquet', columns=['entity_id', 'country', 'cardinality_bucket'])
if 0 < eval_limit < len(all_holdout):
    fraction = eval_limit / len(all_holdout)
    pieces = []
    for _, stratum in all_holdout.groupby(['country', 'cardinality_bucket'], sort=True):
        n = min(len(stratum), max(1, round(len(stratum) * fraction)))
        pieces.append(stratum.sample(n=n, random_state=0))
    eval_holdout = pd.concat(pieces, ignore_index=True)
    evaluation_scope = 'stratified_pilot'
else:
    eval_holdout = all_holdout
    evaluation_scope = 'full_holdout'
holdout_ids = eval_holdout.entity_id.tolist()
s1_country = dict(zip(eval_holdout.entity_id, eval_holdout.country))
print(f'{evaluation_scope}: {len(holdout_ids):,} Source-1 entities; batch_size={batch_size}; max_token_df={max_token_df}')

gt = pd.read_csv(DATA / 'train/train_ground_truth.tsv', sep='\t', dtype='string', keep_default_na=False)
gt = gt[gt.source1_entity_id.isin(set(holdout_ids))].copy()
gt['matched_entity_ids'] = gt['matched_entity_ids'].fillna('')
positive = gt.assign(candidate_id=gt.matched_entity_ids.str.split(',')).explode('candidate_id')
positive = positive[positive.candidate_id.ne('')][['source1_entity_id', 'candidate_id']]
truth = positive.groupby('source1_entity_id')['candidate_id'].agg(set).to_dict()
covered = {s1_id: {k: 0 for k in ks} for s1_id in holdout_ids}
positive_by_country = positive.assign(country=positive.source1_entity_id.map(s1_country)).groupby('country').size().to_dict()
covered_by_country = {country: {k: 0 for k in ks} for country in positive_by_country}
total_ranked = 0
s1_with_candidates = set()
t0 = time.time()
for start in range(0, len(holdout_ids), batch_size):
    ids = holdout_ids[start:start + batch_size]
    con.register('batch_ids_df', pd.DataFrame({'entity_id': ids}))
    try:
        con.execute("CREATE OR REPLACE TEMP TABLE batch_ids AS SELECT * FROM batch_ids_df")
        con.execute("""CREATE OR REPLACE TEMP TABLE batch_ref_tokens AS
        SELECT h.entity_id AS source1_entity_id, h.country, 'name' AS field, token
        FROM (SELECT h.* FROM holdout_refs h JOIN batch_ids b USING (entity_id)) h,
             UNNEST(string_split(h.name_key, chr(31))) AS u(token)
        WHERE token <> ''
        UNION ALL
        SELECT h.entity_id AS source1_entity_id, h.country, 'address' AS field, token
        FROM (SELECT h.* FROM holdout_refs h JOIN batch_ids b USING (entity_id)) h,
             UNNEST(string_split(h.address_key, chr(31))) AS u(token)
        WHERE token <> ''
        """)
        ranked = con.execute("""WITH hits AS (
          SELECT r.source1_entity_id, x.entity_id AS candidate_id,
                 SUM(LN((cc.n + 1.0) / (ts.df + 1.0))) AS score
          FROM batch_ref_tokens r
          JOIN token_stats ts USING (country, field, token)
          JOIN country_counts cc USING (country, field)
          JOIN retrieval_tokens x USING (country, field, token)
          WHERE ts.df <= ?
          GROUP BY r.source1_entity_id, x.entity_id
        ), ranked AS (
          SELECT source1_entity_id, candidate_id,
                 ROW_NUMBER() OVER (PARTITION BY source1_entity_id ORDER BY score DESC, candidate_id) AS rank
          FROM hits
        ) SELECT * FROM ranked WHERE rank <= 100
        """, [max_token_df]).fetch_df()
    finally:
        con.unregister('batch_ids_df')
    total_ranked += len(ranked)
    s1_with_candidates.update(ranked.source1_entity_id.tolist())
    for s1_id, group in ranked.groupby('source1_entity_id'):
        true = truth.get(s1_id, set())
        country = s1_country[s1_id]
        for k in ks:
            hit_count = len(true & set(group.loc[group['rank'] <= k, 'candidate_id']))
            covered[s1_id][k] = hit_count
            covered_by_country[country][k] += hit_count
    if start == 0 or (start // batch_size) % 10 == 0:
        print(f'batches through {start + len(ids):,}/{len(holdout_ids):,}; elapsed={time.time()-t0:.1f}s')
positive_total = len(positive)
recall_at_k = []
for k in ks:
    retained = sum(v[k] for v in covered.values())
    recall_at_k.append({'k': k, 'positive_pairs': positive_total, 'retained': retained, 'recall': retained / max(1, positive_total)})
recall_by_country = {
    country: {str(k): counts[k] / max(1, positive_by_country[country]) for k in ks}
    for country, counts in covered_by_country.items()
}
result = {
    'stage': 'S4-v4-batched-idf',
    'evaluation_scope': evaluation_scope,
    'evaluated_source1_rows': len(holdout_ids),
    'batch_size': batch_size,
    'candidate_stats': {
        'ranked_rows': total_ranked,
        's1_with_candidates': len(s1_with_candidates),
        'avg_top100_per_s1': total_ranked / max(1, len(holdout_ids)),
    },
    'recall_at_k': recall_at_k,
    'recall_by_country': recall_by_country,
    'config': {
        'fields': ['name', 'address'],
        'idf': True,
        'max_token_df': max_token_df,
        'max_temp_directory_size': '4GB',
    },
}
(ART / 'nb04_v4_recall_summary.json').write_text(json.dumps(result, indent=2) + '\n')
run_record = {
    'ts': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'stage': result['stage'],
    'notebook': 'NB04_blocking_v4',
    'config': result['config'] | {'evaluation_scope': evaluation_scope, 'batch_size': batch_size},
    'metrics': {'recall_at_k': recall_at_k, 'candidate_stats': result['candidate_stats']},
    'notes': 'Batched retrieval; no all-holdout join is materialised.',
}
with (ART / 'runs.jsonl').open('a', encoding='utf-8') as f:
    f.write(json.dumps(run_record, sort_keys=True) + '\n')
print(json.dumps(result, indent=2))